In [1]:
#Você pode por favor preparar um função em python, que recebe o caminho de uma imagem (pode ser um .tif ou um .jp2),
#um tipo de compressão (LZW, DEFLATE, ZSTD, pode ser arquivo descomprimido também), os parametros dessa compressão (quando existirem)
#e um tamanho de bloco (128, 256, 512) me gere a mesma imagem em um arquivo .tif de saída,
#que por opção pode ou não estar no formato cloud optimized geotiff.

#Voce pode fazer por partes, comece com o mais simples, primeiro uma funcao q recebe o caminho da imagem e um tipo de compressao
#e gera a imagem saida, dps vc complementa passando os parametros de compressao, dps o tipo de blocagem, dps a geracao de COG.

In [14]:
from osgeo import gdal

arquivo_original = 'https://data.inpe.br/bdc/data/CB4A-MUX-L4-SR/2026_06/CBERS_4A_MUX_RAW_2026_06_19.13_17_01_ETC2/213_113_0/4_BC_UTM_WGS84/CBERS_4A_MUX_20260619_213_113_L4_BAND5_GRID_SURFACE.tif'

#função que recebe os parâmetros para todas as compressões
def converter_tiff(entrada: str, saida: str, compressao: str = None, params_compressao: dict = None, tamanho_bloco: int = 256, como_cog: bool = False):

    dataset = gdal.Open(entrada)
    compressao = [
                    "COMPRESS=ZSTD",
                    "ZSTD_LEVEL=9",
                    "TILED=YES"
    ]
    print(f"Convertendo {entrada}...")
    driver = gdal.GetDriverByName("GTiff")
    dataset_saida = driver.CreateCopy(saida, dataset, options=compressao)
    if dataset_saida:
        print(f"Sucesso! Arquivo salvo em: {saida}")
        dataset = None
        dataset_saida = None
        return True

arquivo_compactado = '/home/jovyan/Downloads/resultado_zstd.tif'

converter_tiff(arquivo_original, arquivo_compactado)
        

Convertendo https://data.inpe.br/bdc/data/CB4A-MUX-L4-SR/2026_06/CBERS_4A_MUX_RAW_2026_06_19.13_17_01_ETC2/213_113_0/4_BC_UTM_WGS84/CBERS_4A_MUX_20260619_213_113_L4_BAND5_GRID_SURFACE.tif...
Sucesso! Arquivo salvo em: /home/jovyan/Downloads/resultado_zstd.tif


True

In [15]:
params_zstd = {'ZSTD_LEVEL': 9, 'PREDICTOR': 2}

converter_tiff(
    entrada = 'https://data.inpe.br/bdc/data/CB4A-MUX-L4-SR/2026_06/CBERS_4A_MUX_RAW_2026_06_19.13_17_01_ETC2/213_113_0/4_BC_UTM_WGS84/CBERS_4A_MUX_20260619_213_113_L4_BAND5_GRID_SURFACE.tif',
    saida = "resultado_cog_zstd.tif",
    compressao = "ZSTD",
    params_compressao = params_zstd,
    tamanho_bloco = 512,
    como_cog = True
    )

Convertendo https://data.inpe.br/bdc/data/CB4A-MUX-L4-SR/2026_06/CBERS_4A_MUX_RAW_2026_06_19.13_17_01_ETC2/213_113_0/4_BC_UTM_WGS84/CBERS_4A_MUX_20260619_213_113_L4_BAND5_GRID_SURFACE.tif...
Sucesso! Arquivo salvo em: resultado_cog_zstd.tif


True

In [16]:
converter_tiff(
    entrada = 'https://data.inpe.br/bdc/data/CB4A-MUX-L4-SR/2026_06/CBERS_4A_MUX_RAW_2026_06_19.13_17_01_ETC2/213_113_0/4_BC_UTM_WGS84/CBERS_4A_MUX_20260619_213_113_L4_BAND5_GRID_SURFACE.tif',
    saida = "resultado_lzw.tif",
    compressao = "LZW",
    tamanho_bloco = 256,
    como_cog = False
    )

Convertendo https://data.inpe.br/bdc/data/CB4A-MUX-L4-SR/2026_06/CBERS_4A_MUX_RAW_2026_06_19.13_17_01_ETC2/213_113_0/4_BC_UTM_WGS84/CBERS_4A_MUX_20260619_213_113_L4_BAND5_GRID_SURFACE.tif...
Sucesso! Arquivo salvo em: resultado_lzw.tif


True

In [17]:
converter_tiff(
    entrada= 'https://data.inpe.br/bdc/data/CB4A-MUX-L4-SR/2026_06/CBERS_4A_MUX_RAW_2026_06_19.13_17_01_ETC2/213_113_0/4_BC_UTM_WGS84/CBERS_4A_MUX_20260619_213_113_L4_BAND5_GRID_SURFACE.tif',
    saida = "resultado_sem_compressao.tif",
    compressao = None,
    tamanho_bloco = 128,
    como_cog = False
    )

Convertendo https://data.inpe.br/bdc/data/CB4A-MUX-L4-SR/2026_06/CBERS_4A_MUX_RAW_2026_06_19.13_17_01_ETC2/213_113_0/4_BC_UTM_WGS84/CBERS_4A_MUX_20260619_213_113_L4_BAND5_GRID_SURFACE.tif...
Sucesso! Arquivo salvo em: resultado_sem_compressao.tif


True

In [18]:
def converter_tiff(entrada, saida, params_compressao=None, tamanho_bloco=512, como_cog=True):
    
    dataset = gdal.Open(entrada)  # Abre o dataset de entrada
    
    # Se o usuário escolher COG, usará o driver COG do GDAL, caso contrário, GTiff tradicional
    driver_nome = "COG" if como_cog else "GTiff"
    driver = gdal.GetDriverByName(driver_nome)
    
    # Configurações padrão de bloco (Tiling)
    gdal_options = [
        "TILED=YES",
        f"BLOCKSIZE={tamanho_bloco}"
    ]
    
    # Se foram passados parâmetros de compressão, adicionamos na lista do GDAL
    if params_compressao:
        for chave, valor in params_compressao.items():
            gdal_options.append(f"{chave}={valor}")
            
    print(f"Convertendo para {driver_nome} com as opções: {gdal_options}...")
    
    # Executa a cópia/conversão do arquivo
    dataset_saida = driver.CreateCopy(saida, dataset, options=gdal_options)
    
    if dataset_saida:
        print(f"Sucesso! Arquivo salvo em: {saida}")
        # Fecha os arquivos para garantir que os dados foram gravados no disco
        dataset = None
        dataset_saida = None
        return True
    else:
        print("Falha na conversão.")
        return False

compressao= {
    'COMPRESS': 'ZSTD',
    'ZSTD_LEVEL': '9',
    'PREDICTOR': '2'
}

converter_tiff(
    entrada = 'https://data.inpe.br/bdc/data/CB4A-MUX-L4-SR/2026_06/CBERS_4A_MUX_RAW_2026_06_19.13_17_01_ETC2/213_113_0/4_BC_UTM_WGS84/CBERS_4A_MUX_20260619_213_113_L4_BAND5_GRID_SURFACE.tif',
    saida = 'resultado_cog_zstd.tif',
    params_compressao = compressao,
    tamanho_bloco = 512,
    como_cog = True
)

Convertendo para COG com as opções: ['TILED=YES', 'BLOCKSIZE=512', 'COMPRESS=ZSTD', 'ZSTD_LEVEL=9', 'PREDICTOR=2']...


Warning 6: driver COG does not support creation option TILED
Warning 6: driver COG does not support creation option ZSTD_LEVEL


Sucesso! Arquivo salvo em: resultado_cog_zstd.tif


True